In [ ]:
# ============================================================
# BoneBot - NHANES 2013-2014 training pipeline
#   Lightweight : P(osteoporosis) triage   (logistic regression)
#   Heavyweight : estimated T-score         (ridge regression)
#
# Lightweight front-gate questions (age + menopause drive the triage):
#  1. "Were you assigned female at birth?"                         (yes/no)
#  2. "How old are you?"                                           (number)
#  3. "Have your periods stopped for good (menopause)?"            (yes/no)
#  4. "Already diagnosed / had a bone scan / take bone meds?"      (yes/no)
# ============================================================

import ssl, urllib.request, io
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score

ssl._create_default_https_context = ssl._create_unverified_context

# --- 1. download + merge (all one row per person -> safe to merge on SEQN) ---
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def load(f):
    req = urllib.request.Request(BASE + f + ".xpt", headers=HEADERS)
    with urllib.request.urlopen(req) as r:
        return pd.read_sas(io.BytesIO(r.read()), format="xport")

files = ["DEMO_H", "BMX_H", "DXXFEM_H", "RHQ_H", "OSQ_H", "SMQ_H",
         "ALQ_H", "MCQ_H", "VID_H", "BIOPRO_H", "PAQ_H", "CBC_H"]
frames = {}
for f in files:
    try:
        frames[f] = load(f)
    except Exception as e:
        print("MISSING / renamed:", f, e)

merged = frames["DEMO_H"]
for f in files[1:]:
    if f in frames:
        merged = merged.merge(frames[f], on="SEQN", how="left")

REF_MEAN, REF_SD = 0.858, 0.120   # NHANES III young-adult white-female femoral-neck reference

# --- 2. heavyweight sample: women 50+ with a femur DXA scan ---
women = merged[(merged.RIAGENDR == 2) & (merged.RIDAGEYR >= 50) & (merged.DXXNKBMD.notna())].copy()

women["Tscore"]                 = (women["DXXNKBMD"] - REF_MEAN) / REF_SD      # target
women["age"]                    = women["RIDAGEYR"]
women["bmi"]                    = women["BMXBMI"]
women["vitaminD"]               = women["LBXVIDMS"]                            # nmol/L
women["calcium"]                = women["LBDSCASI"]                            # mmol/L (weak predictor)
women["alp"]                    = women["LBXSAPSI"]                            # verify var name
women["rbc"]                    = women["LBXRBCSI"]                            # verify var name (exploratory)
women["currentSmoker"]          = women["SMQ040"].isin([1, 2]).astype(int)
women["priorFragilityFracture"] = women[["OSQ010A","OSQ010B","OSQ010C"]].eq(1).any(axis=1).astype(int)
women["glucocorticoids"]        = (women["OSQ130"] == 1).astype(int)
women["highAlcohol"]            = (women["ALQ130"] >= 3).astype(int)
women["yearsSinceMenopause"]    = (women["RIDAGEYR"] - women["RHQ060"]).clip(lower=0)

# wearable activity: mean daily MIMS from the wrist accelerometer, normalised 0-1
pax = load("PAXDAY_H")                                                          # day-level file (not the huge minute file)
act = pax.groupby("SEQN")["PAXMTSD"].mean().rename("daily_mims").reset_index()  # verify PAXMTSD
women = women.merge(act, on="SEQN", how="left")                                 # aggregated first -> no duplicate rows
women["activity_level"] = (women["daily_mims"] / women["daily_mims"].quantile(0.95)).clip(0, 1)

print("heavyweight sample:", women.shape[0], "women")

# --- 3. lightweight triage: P(osteoporosis) from age + menopause (BROAD age range) ---
fem = merged[(merged.RIAGENDR == 2) & (merged.DXXNKBMD.notna()) & (merged.RIDAGEYR >= 18)].copy()
fem["osteoporosis"]   = ((fem["DXXNKBMD"] - REF_MEAN) / REF_SD <= -2.5).astype(int)   # 1 = yes
fem["age"]            = fem["RIDAGEYR"]
fem["postmenopausal"] = ((fem["RIDAGEYR"] - fem["RHQ060"]) >= 0).astype(int)          # missing menopause age -> 0

Xl, yl = fem[["age", "postmenopausal"]], fem["osteoporosis"]
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(Xl, yl, test_size=0.25, random_state=0, stratify=yl)
triage = LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l)
triage_auc = roc_auc_score(yte_l, triage.predict_proba(Xte_l)[:, 1])

def prob_osteoporosis(age, postmenopausal):
    row = pd.DataFrame([[age, postmenopausal]], columns=["age", "postmenopausal"])
    return triage.predict_proba(row)[0, 1]        # column 1 = P(osteoporosis)

# --- 4. heavyweight: estimated T-score (ridge) ---
features = ["age", "bmi", "vitaminD", "calcium", "currentSmoker", "priorFragilityFracture",
            "glucocorticoids", "highAlcohol", "yearsSinceMenopause", "alp", "rbc", "activity_level"]

X = women[features].apply(lambda c: c.fillna(c.median()))    # median-impute missing
y = women["Tscore"]
Xtr_h, Xte_h, ytr_h, yte_h = train_test_split(X, y, test_size=0.25, random_state=0)
model = Ridge(alpha=1.0).fit(Xtr_h, ytr_h)
pred_h = model.predict(Xte_h)

perf_table = pd.DataFrame({
    "Metric": ["MAE", "R2", "n_train", "n_test"],
    "Value":  [round(mean_absolute_error(yte_h, pred_h), 3),
               round(r2_score(yte_h, pred_h), 3), len(Xtr_h), len(Xte_h)],
})
coef_table = pd.DataFrame({"Term": ["intercept"] + features,
                           "Coefficient": [model.intercept_] + list(model.coef_)})
coef_table["Coefficient"] = coef_table["Coefficient"].round(4)
coef_table["Direction"]   = coef_table["Coefficient"].apply(
    lambda c: "raises T-score" if c > 0 else "lowers T-score")
coef_table = coef_table.sort_values("Coefficient").reset_index(drop=True)

# --- 5. results ---
print(f"\nLIGHTWEIGHT triage - AUC: {triage_auc:.3f}")
for a, m in [(23, 0), (43, 0), (55, 1), (70, 1)]:
    print(f"   age {a}, postmeno {m}: P(osteoporosis) = {prob_osteoporosis(a, m):.1%}")

print("\nHEAVYWEIGHT T-score model")
print(perf_table.to_string(index=False))
print()
print(coef_table.to_string(index=False))

# --- 6. benchmark: are we better than the tools doctors use today? ---
# Honest framing: we do NOT compete with a DXA scan (the ground truth). We compete
# with the tools used to decide WHO needs a scan — OST and clinical-risk-factors-only
# (a FRAX proxy) — evaluated on the SAME held-out women.

# (a) OST (Osteoporosis Self-assessment Tool): 0.2*(weight_kg - age); lower = higher risk
test = fem.loc[Xte_l.index].copy()
test["ost"] = 0.2 * (test["BMXWT"] - test["age"])
m = test["ost"].notna()
ost_auc = roc_auc_score(test.loc[m, "osteoporosis"], -test.loc[m, "ost"])   # -ost so higher = more risk

# (b) clinical-risk-factors-only model vs full model (+ labs + wearable), same train/test split
clinical = ["age", "bmi", "priorFragilityFracture", "glucocorticoids",
            "currentSmoker", "highAlcohol", "yearsSinceMenopause"]
Xc = women[clinical].apply(lambda c: c.fillna(c.median()))
clin_model = Ridge(alpha=1.0).fit(Xc.loc[Xtr_h.index], ytr_h)               # same rows as the full model
clin_mae = mean_absolute_error(yte_h, clin_model.predict(Xc.loc[Xte_h.index]))
full_mae = mean_absolute_error(yte_h, pred_h)

print("=== Better than current practice? (same held-out women) ===")
print(f"Triage AUC  - OST tool:            {ost_auc:.3f}")
print(f"Triage AUC  - our model:           {triage_auc:.3f}   <- higher is better")
print(f"T-score MAE - clinical only:       {clin_mae:.3f}")
print(f"T-score MAE - + labs + wearable:   {full_mae:.3f}   <- lower is better")
print(f"\nAdded value from objective data:   {clin_mae - full_mae:+.3f} MAE improvement")